# E.2 Sickle-cell trait
How can HbS persist when it causes severe disease in homozygotes?

Let pS be the HbS frequency before selection. Under random mating, AA, AS and SS have probabilities (1-pS)², 2pS(1-pS) and pS². Their fitnesses predict
$$\bar w=(1-p_S)^2w_{AA}+2p_S(1-p_S)w_{AS}+p_S^2w_{SS},$$
$$p_{S,\mathrm{sel}}=
\frac{p_S^2w_{SS}+p_S(1-p_S)w_{AS}}{\bar w}.$$
A finite next generation would then draw Binomial(2Ne, pS,sel). Here we first iterate the deterministic update and compare environments.


In [ ]:
library(hpgR)
options(repr.plot.width = 9, repr.plot.height = 3.8)
drawSelectionFigure <- function(name, draw, height = 3.6) {
    draw()
    figureDir <- Sys.getenv("P8149_FIGURE_DIR")
    if (nzchar(figureDir)) {
        dir.create(figureDir, recursive = TRUE, showWarnings = FALSE)
        pdf(file.path(figureDir, paste0(name, ".pdf")), width = 9,
            height = height, family = "Helvetica", pointsize = 12,
            useDingbats = FALSE)
        tryCatch(draw(), finally = dev.off())
    }
}


## Hold the starting allele fixed and change the fitnesses
Start at pS=0.02. The malaria setting uses (AA, AS, SS)=(0.92, 1, 0.25); the other setting uses (1, 0.98, 0.25). These are teaching values. With heterozygote advantage the internal equilibrium is
$$p_{S,*}=\frac{1-w_{AA}}{(1-w_{AA})+(1-w_{SS})}.$$
The function expects SS, AS, AA because we are tracking S.


In [ ]:
generations <- 150L
malaria <- c(AA = .92, AS = 1, SS = .25)
noMalaria <- c(AA = 1, AS = .98, SS = .25)
equilibrium <- balancedFrequency(rev(malaria))
withMalaria <- selectionFrequencies(.02, rev(malaria), generations)
withoutMalaria <- selectionFrequencies(.02, rev(noMalaria), generations)
drawSelectionFigure("selection_sickle_environment", function() {
    par(mar = c(4.2, 4.3, 1.2, .8))
    plot(0:generations, withMalaria, type = "l", lwd = 2, col = "#B33B32",
         ylim = c(0, .13), xlab = "Generation", ylab = "Deterministic HbS frequency")
    lines(0:generations, withoutMalaria, lwd = 2, col = "#306C9A")
    abline(h = equilibrium, lty = 2, col = "#B33B32")
    legend("right", c("Malaria setting", "Other setting", "Equilibrium"),
           col = c("#B33B32", "#306C9A", "#B33B32"), lty = c(1, 1, 2), bty = "n")
})


In [ ]:
stopifnot(withMalaria[2] > .02, withoutMalaria[2] < .02,
          abs(selectedFrequency(equilibrium, rev(malaria)) - equilibrium) < 1e-12)
c(initial = .02, malariaEquilibrium = equilibrium)


## Connect the fitness model to observed malaria protection
At the Mali site in the MalariaGEN study, HbAS occurred in 4 of 453 severe-malaria cases and 28 of 344 controls. These counts measure one clinical outcome. Filling the genotype-fitness table also requires survival and reproductive contribution over the life course.


In [ ]:
library(hpgR)

count <- matrix(
  c(4, 449, 28, 316),
  nrow = 2, byrow = TRUE,
  dimnames = list(group = c('Severe malaria', 'Control'),
                  genotype = c('HbAS', 'Non-HbAS'))
)

oddsRatio <- (count[1, 'HbAS'] / count[1, 'Non-HbAS']) /
  (count[2, 'HbAS'] / count[2, 'Non-HbAS'])
hbASFraction <- count[, 'HbAS'] / rowSums(count)

stopifnot(sum(count[1, ]) == 453, sum(count[2, ]) == 344)
c(raw_oddsRatio = oddsRatio, hb_as_cases = hbASFraction[1],
  hb_as_controls = hbASFraction[2])

In [ ]:
barplot(hbASFraction, col = c('#B44A3A', '#306C9A'), ylim = c(0, 0.10),
        ylab = 'HbAS fraction', names.arg = names(hbASFraction))
mtext(sprintf('Raw odds ratio = %.2f', oddsRatio), side = 3, line = 0.2)

## Why the odds ratio does not determine an equilibrium

For the idealized heterozygote-advantage model $w_{AA}=1-s_A$, $w_{AS}=1$, and $w_{SS}=1-s_S$, the HbS equilibrium is $p_S^*=s_A/(s_A+s_S)$. The case–control table does not provide either selection coefficient. Two teaching fitness tables can therefore agree that HbAS has the highest fitness and still imply different equilibria.


In [ ]:
fitnessModel <- data.frame(
  model = c('Model 1', 'Model 2'),
  wAA = c(0.90, 0.60),
  wAS = c(1.00, 1.00),
  wSS = c(0.20, 0.20)
)
fitnessModel$pSEquilibrium <- apply(
  fitnessModel[, c("wSS", "wAS", "wAA")], 1, balancedFrequency
)
fitnessModel

The two equilibrium values differ because the unmeasured costs differ. They are not estimates from Mali; they demonstrate the missing information.

## What to remember

The raw odds ratio is about 0.10 and supports strong protection against severe malaria in this sample. Balancing selection needs a complete genotype-fitness model. A clinical odds ratio cannot be inserted into that model as a fitness value.

**Sources.** Malaria Genomic Epidemiology Network (2014), *Nature Genetics* 46:1197–1204, Supplementary Table 11; Pritchard, *An Owner's Guide to the Human Genome*, Section 2.6. The odds ratio is an unadjusted calculation from one site; the fitness tables are teaching values, not population estimates.
